In [ ]:
import random
import sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np

# has to be here for inports to work in vscode
project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.append(project_root)
from animal_recognition.src.data.augmentations import get_train_transforms, _MEAN, _STD


def denormalize(tensor):

    img = tensor.numpy().transpose(1, 2, 0)

    mean = np.array(_MEAN)
    std = np.array(_STD)

    img = std * img + mean

    img = np.clip(img, 0, 1)
    return img


IMAGE_FOLDER = Path("../../animal_recognition/data/processed/accepted")
NUM_IMAGES = 10
NUM_AUGS = 10
IMAGE_SIZE = 224

valid_exts = {".jpg", ".jpeg", ".png"}
image_paths = [p for p in IMAGE_FOLDER.rglob("*") if p.is_file() and p.suffix.lower() in valid_exts]

if not image_paths:
    print(f"No images found in {IMAGE_FOLDER}")
else:
    print(f"Found {len(image_paths)} images. Generating preview...")

    selected_paths = random.sample(image_paths, min(NUM_IMAGES, len(image_paths)))

    transform = get_train_transforms(image_size=IMAGE_SIZE)

    fig, axes = plt.subplots(
        len(selected_paths), NUM_AUGS + 1, figsize=(3 * (NUM_AUGS + 1), 3 * len(selected_paths))
    )

    if len(selected_paths) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, img_path in enumerate(selected_paths):
        original_img = cv2.imread(str(img_path))
        original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)

        ax = axes[i, 0]
        ax.imshow(original_img)
        ax.set_title("Original")
        ax.axis("off")

        for j in range(1, NUM_AUGS + 1):
            augmented_tensor = transform(image=original_img)["image"]

            vis_img = denormalize(augmented_tensor)

            ax = axes[i, j]
            ax.imshow(vis_img)
            ax.set_title(f"Augmented {j}")
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import sys
from pathlib import Path
import random
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path.cwd().parent.parent
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from animal_recognition.src.data.dataset import AnimalDataset, CLASSES

data_dir = PROJECT_ROOT / "animal_recognition" / "data" / "processed" / "accepted"
dataset = AnimalDataset(root=data_dir, transform=None)

class_indices = {i: [] for i in range(len(CLASSES))}
for idx, (_, label) in enumerate(dataset.samples):
    if label in class_indices:
        class_indices[label].append(idx)

sampled_data = {}
for label, indices in class_indices.items():
    if not indices:
        continue
    sampled_data[label] = random.sample(indices, min(5, len(indices)))

num_rows = 10
images_per_class = 5
num_cols = 13  

width_ratios = [1.5, 1, 1, 1, 1, 1, 1.0, 1.5, 1, 1, 1, 1, 1]

fig, axes = plt.subplots(
    num_rows, 
    num_cols, 
    figsize=(22, 14), 
    gridspec_kw={'width_ratios': width_ratios}, 
    squeeze=False
)
fig.suptitle("Animal Dataset Sample Overview", fontsize=20, y=0.98)

for r in range(num_rows):
    for c in range(num_cols):
        axes[r, c].axis("off")

for i, (label, sampled_indices) in enumerate(sorted(sampled_data.items())):
    class_name = CLASSES[label]
    
    if i < 10:
        row_idx = i
        text_col = 0
        img_start_col = 1
    else:
        row_idx = i - 10
        text_col = 7  # Shifted over to account for the spacer at column 6
        img_start_col = 8
        
    text_ax = axes[row_idx, text_col]
    text_ax.text(1.0, 0.5, f"{class_name}\n({label})", 
                 transform=text_ax.transAxes, fontsize=12, 
                 va='center', ha='right', weight='bold')
        
    for img_idx in range(images_per_class):
        ax = axes[row_idx, img_start_col + img_idx]
        
        if img_idx < len(sampled_indices):
            dataset_idx = sampled_indices[img_idx]
            img_path, img_label = dataset.samples[dataset_idx]
            
            # Load full image without cropping
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)

plt.tight_layout()
plt.show()

# Report figures and statistics

Run in order. Figures are written to `report/figures/` as PDF and PNG.

## Setup

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

FIG_DIR = PROJECT_ROOT / "report" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS = PROJECT_ROOT / "results"

C_BLUE, C_ORANGE, C_AQUA = "#2a78d6", "#eb6834", "#1baf7a"
C_VIOLET, C_RED = "#4a3aa7", "#e34948"
INK, INK_MUTED, GRID = "#0b0b0b", "#52514e", "#dcdcd7"
COL_W, PAGE_W = 3.4, 7.0  # one text column / full width of the report

mpl.rcParams.update({
    "figure.dpi": 160, "savefig.dpi": 160, "savefig.bbox": "tight",
    "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
    "axes.edgecolor": INK_MUTED, "axes.labelcolor": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "text.color": INK, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.frameon": False, "legend.fontsize": 7,
    "grid.color": GRID, "grid.linewidth": 0.6, "lines.linewidth": 1.6,
})


def save(fig, name):
    out = FIG_DIR / f"{name}.pdf"
    fig.savefig(out)
    fig.savefig(out.with_suffix(".png"))


def load_results(csv_path):
    df = pd.read_csv(RESULTS / csv_path)
    for col in ("Accuracy", "Precision", "Recall", "F1"):
        df[col] = df[col].str.rstrip("%").astype(float)
    return df


def parse_run_config(filenames):
    rows = []
    for f in filenames:
        stem = f.split("/")[-1]
        sweep = re.search(
            r"experiment_(base|smaller_warmup_steplr|smaller_warmup)_([0-9.e\-]+)_pre", stem)
        rows.append(dict(
            arch="GCViT" if "gcvit" in stem else "ConvNeXtV2",
            size=re.search(r"^(?:gcvit|convnextv2)_(tiny|small|base)", stem).group(1),
            logged_lr=float(re.search(r"_lr([0-9.e\-+]+)_wd", stem).group(1)),
            weight_decay=float(re.search(r"_wd([0-9.e\-+]+)_ls", stem).group(1)),
            label_smoothing=float(re.search(r"_ls([0-9.e\-+]+)_sz", stem).group(1)),
            batch_size=int(re.search(r"_bs(\d+)_", stem).group(1)),
            pretrained="preTrue" in stem,
            scheduler="Cosine" if "CosineLR" in stem else "StepLR",
            augmentation=re.search(r"_aug(\w+?)_sched", stem).group(1),
            sweep_variant=sweep.group(1) if sweep else None,
            sweep_peak_lr=float(sweep.group(2)) if sweep else np.nan,
        ))
    return pd.DataFrame(rows)


def with_config(df):
    cfg = parse_run_config(df["File"])
    cfg.index = df.index
    return pd.concat([df, cfg], axis=1)


FAMILY = {
    "Initial_Experiments": "Fine-tuning strategies (tiny)",
    "Initial_Experiments_Top_5": "Scaled to base (1/sqrt(N) rule)",
    "Initial_Experiments_Top_5_Failed": "Scaled to base (muP width rule)",
    "BitFit_Tiny": "BitFit LR sweep (tiny)",
    "BitFit_Base": "BitFit LR sweep (base)",
    "Pretrained_False": "From scratch",
}

ours = with_config(load_results("results_our_test_dataset.csv"))
provided = with_config(load_results("results_provided_dataset.csv"))
both = ours.merge(provided, on="File", suffixes=("_ours", "_prov"))
for _df in (ours, provided):
    _df["family"] = _df["Folder"].map(FAMILY)
both["family"] = both["Folder_ours"].map(FAMILY)

N_ALL, N_PRE = len(ours), int(ours.pretrained.sum())
print(f"{N_ALL} checkpoints, {N_PRE} pretrained")

## Headline numbers

In [ ]:
from scipy.stats import binomtest, pearsonr, spearmanr

pre = both[both.pretrained_ours].copy()
pre["gap"] = pre.Accuracy_ours - pre.Accuracy_prov

best_ours = both.loc[both.F1_ours.idxmax()]
best_prov = both.loc[both.Accuracy_prov.idxmax()]
rank_ours = both.Accuracy_ours.rank(ascending=False)
rank_prov = both.Accuracy_prov.rank(ascending=False)

rho_all = spearmanr(both.Accuracy_ours, both.Accuracy_prov).statistic
rho_pre = spearmanr(pre.Accuracy_ours, pre.Accuracy_prov).statistic
r_pre = pearsonr(pre.Accuracy_ours, pre.Accuracy_prov).statistic

cols = ["Accuracy", "F1", "arch", "size", "family"]
print("top 5 on our test set (by F1)")
print(ours.nlargest(5, "F1")[cols].to_string(index=False))
print("\ntop 5 on the provided set (by accuracy)")
print(provided.nlargest(5, "Accuracy")[cols].to_string(index=False))

print("\npretrained vs from scratch")
print(pd.concat(
    {"ours": ours.groupby("pretrained")[["Accuracy", "F1"]].agg(["count", "min", "max", "mean"]),
     "provided": provided.groupby("pretrained")[["Accuracy", "F1"]].agg(["count", "min", "max", "mean"])}
).round(2).to_string())

print(f"\ngap ours-provided (pretrained): mean {pre.gap.mean():.2f}"
      f"  IQR {pre.gap.quantile(.25):.2f}-{pre.gap.quantile(.75):.2f}")
print(f"spearman {rho_all:.2f} (all {len(both)}) / {rho_pre:.2f} (pretrained {len(pre)})"
      f", pearson {r_pre:.2f}")
print(f"best on ours -> rank {rank_prov[best_ours.name]:.0f} on provided; "
      f"best on provided -> rank {rank_ours[best_prov.name]:.0f} on ours")
for n, acc in ((1991, best_ours.Accuracy_ours), (143, best_prov.Accuracy_prov)):
    ci = binomtest(round(acc / 100 * n), n).proportion_ci(0.95)
    print(f"n={n:<5d} {acc:.2f}%  95% CI [{ci.low*100:.1f}, {ci.high*100:.1f}]")

pre_ours = ours[ours.pretrained]
print("\nby architecture and size")
print(pre_ours.groupby(["arch", "size"])[["Accuracy", "F1"]].agg(["count", "mean", "max"]).round(2).to_string())
print("\nby schedule")
print(pre_ours.groupby("scheduler")[["Accuracy", "F1"]].agg(["count", "mean", "max"]).round(2).to_string())

scratch = both[~both.pretrained_ours].copy()
scratch["run"] = [Path(f).stem.split("_preFalse")[0].split("_", 2)[2] for f in scratch.File]
print("\nfrom scratch, matched augmentation pairs")
print(scratch.pivot_table(index=["size_ours", "run"], columns="augmentation_ours",
                          values=["Accuracy_ours", "Accuracy_prov"]).round(2).to_string())
print(scratch.groupby("augmentation_ours")[["Accuracy_ours", "Accuracy_prov", "F1_ours"]].mean().round(2).to_string())

## Sanitisation, per class

In [ ]:
from animal_recognition.src.data.dataset import CLASSES

RAW = PROJECT_ROOT / "animal_recognition/data/raw"
ACC = PROJECT_ROOT / "animal_recognition/data/processed/accepted"
EXT = {".jpg", ".jpeg", ".png", ".webp"}


def count_images(root, cls):
    d = root / cls
    return sum(1 for p in d.iterdir() if p.suffix.lower() in EXT) if d.is_dir() else 0


counts = pd.DataFrame({"cls": CLASSES,
                       "raw": [count_images(RAW, c) for c in CLASSES],
                       "accepted": [count_images(ACC, c) for c in CLASSES]})
counts["dropped"] = counts.raw - counts.accepted
counts["drop_pct"] = 100 * counts.dropped / counts.raw
order = counts.sort_values("drop_pct", ascending=False)

print(f"{counts.raw.sum()} scraped -> {counts.accepted.sum()} kept "
      f"({100 * counts.dropped.sum() / counts.raw.sum():.1f}% dropped)")
print(order.head(6).round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(COL_W, 2.15))
bars = ax.barh(np.arange(len(order)), order.drop_pct, height=0.7, color=C_BLUE)
bars[0].set_color(C_ORANGE)
ax.set_yticks(np.arange(len(order)), order.cls, fontsize=6)
ax.invert_yaxis()
ax.set_xlabel("images discarded by the detector pass (%)")
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.annotate(f"{order.drop_pct.iloc[0]:.0f}%  (ImageNet 'tiger cat'\nsynset contains real tigers)",
            xy=(order.drop_pct.iloc[0], 0), xytext=(10.5, 2.4), fontsize=6,
            color=INK_MUTED, va="center")
ax.set_title("Scraped-data sanitisation, per class", loc="left")
save(fig, "fig_sanitisation")
plt.close(fig)

## Detector threshold sweep

In [ ]:
DET = RESULTS
det = pd.concat([pd.read_csv(DET / f) for f in
                 ("yoloworld_results_full.csv", "yoloworld_results_fine_grained.csv",
                  "yoloworld_results_full_more_thresholds.csv")], ignore_index=True)
det["n_prompts"] = det.class_set.apply(lambda s: len(eval(s)))
det["scale"] = det.model_name.str.extract(r"yolov8([smlx])")[0]
det = det.drop_duplicates(["model_name", "n_prompts", "confidence_threshold"])

print(f"{len(det)} configurations: scales {sorted(det.scale.unique())}, "
      f"prompt sets {sorted(det.n_prompts.unique())}, "
      f"{det.confidence_threshold.nunique()} thresholds")

sub = det[(det.n_prompts == 14) & (det.scale == "x")].sort_values("confidence_threshold")
fig, ax = plt.subplots(figsize=(COL_W, 2.1))
ax.plot(sub.confidence_threshold, sub.recall * 100, marker="o", ms=3.5,
        color=C_BLUE, label="Recall (target animal found)")
ax.plot(sub.confidence_threshold, sub.precision * 100, marker="s", ms=3.5,
        color=C_ORANGE, label="Precision")
ax.axvline(0.05, color=INK_MUTED, lw=0.8, ls=":")
ax.annotate("deployed threshold 0.05", xy=(0.058, 99.6), fontsize=6, color=INK_MUTED)
ax.set_xscale("log")
ax.set_xlabel("YOLO-World confidence threshold")
ax.set_ylabel("%")
ax.set_ylim(83, 101.5)
ax.yaxis.grid(True)
ax.set_axisbelow(True)
ax.legend(loc="lower right")
ax.set_title("yolov8x-worldv2, 14 prompts", loc="left")
save(fig, "fig_detector")
plt.close(fig)

print("\nat threshold 0.05, 14 prompts")
print(det[(det.confidence_threshold == 0.05) & (det.n_prompts == 14)]
      [["model_name", "accuracy", "recall", "precision",
        "true positive", "true negative", "false postivie", "false negative"]]
      .round(4).to_string(index=False))

## BitFit learning-rate sweep

In [ ]:
sweep = both[both.sweep_variant_ours.notna()].copy()
VARIANTS = [("base", "Cosine, 10-epoch warm-up", C_BLUE, "o"),
            ("smaller_warmup", "Cosine, 5-epoch warm-up", C_ORANGE, "s"),
            ("smaller_warmup_steplr", "StepLR, 5-epoch warm-up", C_AQUA, "^")]
TRAINABLE = {"tiny": (27.7, 0.236), "base": (89.3, 0.456)}

fig, axes = plt.subplots(1, 2, figsize=(PAGE_W, 1.95), sharey=True)
for ax, size in zip(axes, ("tiny", "base")):
    s = sweep[sweep.size_ours == size]
    for key, label, color, marker in VARIANTS:
        v = s[s.sweep_variant_ours == key].sort_values("sweep_peak_lr_ours")
        ax.plot(v.sweep_peak_lr_ours, v.Accuracy_ours, marker=marker, ms=4,
                color=color, label=label)
    n_par, n_train = TRAINABLE[size]
    ax.axvspan(5e-5, 1.2e-2, color=C_BLUE, alpha=0.07, lw=0)
    ax.set_xscale("log")
    ax.set_xlabel("peak learning rate")
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
    ax.set_title(f"GCViT-{size} ({n_par:.0f}M params, {100 * n_train / n_par:.2f}% trainable)",
                 loc="left")
axes[0].set_ylabel("accuracy on our test set (%)")
axes[0].legend(loc="lower left")
save(fig, "fig_bitfit_sweep")
plt.close(fig)

print("accuracy on our test set (%) by peak LR")
print(sweep.pivot_table(index="sweep_peak_lr_ours",
                        columns=["size_ours", "sweep_variant_ours"],
                        values="Accuracy_ours").round(2).to_string())

## Agreement between the two test sets

In [ ]:
GROUPS = [
    ("Fine-tuning strategies", ["Fine-tuning strategies (tiny)",
                                "Scaled to base (1/sqrt(N) rule)",
                                "Scaled to base (muP width rule)"], C_BLUE, "o"),
    ("BitFit LR sweep", ["BitFit LR sweep (tiny)", "BitFit LR sweep (base)"], C_ORANGE, "s"),
    ("From random init", ["From scratch"], C_AQUA, "^"),
]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(PAGE_W, 2.35))

for label, fams, color, marker in GROUPS:
    s = both[both.family.isin(fams)]
    ax1.scatter(s.Accuracy_ours, s.Accuracy_prov, s=16, marker=marker,
                facecolor=color, edgecolor="white", linewidth=0.5, label=label, zorder=3)
ax1.plot([20, 95], [20, 95], color=INK_MUTED, lw=0.8, ls="--", zorder=1)
ax1.annotate("y = x", xy=(44, 36), fontsize=6.5, color=INK_MUTED, rotation=41)
ax1.legend(loc="upper left")
ax1.set_title(f"(a) all {N_ALL} checkpoints (Spearman $\\rho$ = {rho_all:.2f})", loc="left")

for label, fams, color, marker in GROUPS[:2]:
    s = pre[pre.family.isin(fams)]
    ax2.scatter(s.Accuracy_ours, s.Accuracy_prov, s=16, marker=marker,
                facecolor=color, edgecolor="white", linewidth=0.5, label=label, zorder=3)
for row, tag, xy in ((best_ours, "best on our set", (91.2, 84.0)),
                     (best_prov, "best on provided set", (82.0, 88.6))):
    ax2.scatter([row.Accuracy_ours], [row.Accuracy_prov], s=75, facecolor="none",
                edgecolor=C_VIOLET, linewidth=1.2, zorder=4)
    ax2.annotate(tag, xy=xy, fontsize=6, color=C_VIOLET, ha="left" if xy[0] > 88 else "right")

# Reference cross for the 95% binomial interval on each axis. The provided set
# is 14x smaller, so its bar is ~4x the length of ours; parked in the empty
# upper-left corner, which holds no runs above 80%.
xerr = 1.96 * np.sqrt(0.88 * 0.12 / 1991) * 100
yerr = 1.96 * np.sqrt(0.85 * 0.15 / 143) * 100
ax2.errorbar([71.0], [84.0], xerr=xerr, yerr=yerr,
             color=INK_MUTED, lw=0.9, capsize=2, zorder=2)
ax2.annotate(f"95% binomial sampling error\n$\\pm${xerr:.1f} pp ours ($n$=1,991)\n"
             f"$\\pm${yerr:.1f} pp provided ($n$=143)",
             xy=(73.0, 84.0), fontsize=6, color=INK_MUTED, ha="left", va="center")
ax2.set_xlim(68, 94.5)
ax2.set_ylim(68, 90.5)
ax2.legend(loc="lower right")
ax2.set_title(f"(b) {N_PRE} pretrained runs only ($\\rho$ = {rho_pre:.2f})", loc="left")

for ax in (ax1, ax2):
    ax.set_xlabel("accuracy, our test set (%)")
    ax.set_ylabel("accuracy, provided set (%)")
    ax.grid(True)
    ax.set_axisbelow(True)
save(fig, "fig_transfer")
plt.close(fig)

## Spread by experiment family

In [ ]:
FAM_ORDER = ["From scratch", "BitFit LR sweep (base)", "BitFit LR sweep (tiny)",
             "Fine-tuning strategies (tiny)", "Scaled to base (muP width rule)",
             "Scaled to base (1/sqrt(N) rule)"]
SERIES = [(ours, C_BLUE, "o", "Our test set (n = 1991, no reject class)", -0.16),
          (provided, C_ORANGE, "s", "Provided set (n = 143, reject class active)", 0.16)]

fig, ax = plt.subplots(figsize=(PAGE_W, 2.05))
rng = np.random.default_rng(0)
for df, color, marker, label, off in SERIES:
    for i, fam in enumerate(FAM_ORDER):
        v = df[df.family == fam].Accuracy.values
        ax.scatter(v, i + off + rng.uniform(-0.055, 0.055, len(v)), s=13, marker=marker,
                   facecolor=color, edgecolor="white", linewidth=0.4, zorder=3,
                   label=label if i == 0 else None)
        ax.plot([v.mean()], [i + off], marker="|", ms=13, color=INK, zorder=4)
ax.set_yticks(range(len(FAM_ORDER)),
              [f"{f}\n(n={int((ours.family == f).sum())})" for f in FAM_ORDER], fontsize=6.5)
ax.set_xlabel("accuracy (%)   —   vertical tick marks the family mean")
ax.set_ylim(-0.6, len(FAM_ORDER) - 0.4)
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.legend(loc="upper left")
ax.set_title("Every trained checkpoint, by experiment family", loc="left")
save(fig, "fig_families")
plt.close(fig)

## Per-class error analysis

Needs the checkpoint and a GPU, about a minute. Saves the predictions to `/tmp` so the
plot below can be re-run without repeating inference.

In [ ]:
import tempfile
import time

import torch
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader

import animal_recognition.src.data.augmentations as augmentations
from animal_recognition.src.data.dataset import CLASSES, AnimalDataset
from animal_recognition.src.models.classifier_gcvit import GCViTClassifier
from animal_recognition.src.models.yoloworld import YoloWorldDetector

CKPT = (PROJECT_ROOT / "animal_recognition/models/weights/BitFit_Base/"
        / "gcvit_base_gcvit_base_bitfit_experiment_base_0.01_preTrue_bs32_lr0.001_wd0_ls0.0_sz224_augstronger_schedCosineLRScheduler.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

clf = GCViTClassifier(pretrained=False, model_name="gcvit_base")
clf.load_state_dict(torch.load(CKPT, map_location="cpu"))
clf.eval().to(DEVICE)
transform = augmentations.get_val_transforms(image_size=224)

ds = AnimalDataset(PROJECT_ROOT / "animal_recognition/data/test_dataset", transform=transform)
y_true_a, y_pred_a = [], []
with torch.no_grad():
    for x, y in DataLoader(ds, batch_size=64, num_workers=0):
        y_pred_a += clf(x.to(DEVICE)).argmax(1).cpu().tolist()
        y_true_a += y.tolist()
cm_a = confusion_matrix(y_true_a, y_pred_a, labels=list(range(20)))

detector = YoloWorldDetector()
# Restricted to the accept prompts, as in inference.py. ultralytics caches
# predictor kwargs, so `classes` has to be passed on every call.
ACCEPT_PROMPTS = list(range(detector.reject_classes_index))
img_dir = PROJECT_ROOT / "images"
labels = pd.read_csv(img_dir / "labels.csv")

y_true_b, y_pred_b = [], []
t0 = time.time()
with torch.no_grad():
    for fn, lab in labels[["filename", "label"]].itertuples(index=False):
        image = Image.open(img_dir / fn).convert("RGB")
        with tempfile.NamedTemporaryFile(suffix=".jpg") as tmp:
            image.save(tmp.name)
            crop, _, _ = detector.predict(Path(tmp.name), confidence_threshold=0.05,
                                          reject_on_invalid_class=True, classes=ACCEPT_PROMPTS)
        if crop is None:
            pred = -1
        else:
            x = transform(image=crop[:, :, ::-1])["image"].unsqueeze(0).to(DEVICE)
            pred = int(clf(x).argmax(1).item())
        y_true_b.append(int(lab))
        y_pred_b.append(pred)

lbls = [-1] + list(range(20))
cm_b = confusion_matrix(y_true_b, y_pred_b, labels=lbls)
np.save("/tmp/cm_a.npy", cm_a)
np.save("/tmp/cm_b.npy", cm_b)

yt, yp = np.array(y_true_b), np.array(y_pred_b)
print(f"pipeline on {len(yt)} images in {time.time() - t0:.0f}s -> {(yt == yp).mean() * 100:.2f}% acc")

rep = classification_report(y_true_a, y_pred_a, target_names=CLASSES,
                            output_dict=True, zero_division=0)
per = pd.DataFrame(rep).T.loc[CLASSES].sort_values("recall")
print("\nour test set, per class (sorted by recall)")
print((per[["precision", "recall", "f1-score", "support"]] * [100, 100, 100, 1]).round(1).to_string())

off = [(CLASSES[i], CLASSES[j], cm_a[i, j])
       for i in range(20) for j in range(20) if i != j and cm_a[i, j] > 0]
print("\ntop confusions")
for a, b, n in sorted(off, key=lambda t: -t[2])[:14]:
    print(f"  {n:3d}  {a:>18s} -> {b}")

rep_b = classification_report(y_true_b, y_pred_b, labels=lbls,
                              target_names=["reject(-1)"] + CLASSES,
                              output_dict=True, zero_division=0)
r = rep_b["reject(-1)"]
print(f"\nprovided set: {(yt == -1).sum()} confounders, {((yt == -1) & (yp == -1)).sum()} rejected; "
      f"{(yt != -1).sum()} targets, {((yt != -1) & (yp == -1)).sum()} wrongly rejected; "
      f"{((yt != -1) & (yp != -1) & (yt != yp)).sum()} breed errors")
print(f"reject class P {r['precision'] * 100:.1f} R {r['recall'] * 100:.1f} F1 {r['f1-score'] * 100:.1f}"
      f" | macro F1 {rep_b['macro avg']['f1-score'] * 100:.2f}")

## Confusion matrices

In [ ]:
cm_a = np.load("/tmp/cm_a.npy")
cm_b = np.load("/tmp/cm_b.npy")

fig, (axa, axb) = plt.subplots(1, 2, figsize=(PAGE_W, 2.95))
for ax, cm, names, title in (
    (axa, cm_a, CLASSES, "(a) classifier alone, our test set (n = 1991)"),
    (axb, cm_b, ["reject"] + CLASSES, "(b) full pipeline, provided set (n = 143)"),
):
    row = cm / np.maximum(cm.sum(1, keepdims=True), 1) * 100
    ax.imshow(row, cmap="Blues", vmin=0, vmax=100)
    ax.set_xticks(range(len(names)), names, rotation=90, fontsize=5)
    ax.set_yticks(range(len(names)), names, fontsize=5)
    ax.set_title(title, loc="left")
    ax.set_xlabel("predicted")
    for i in range(len(names)):
        for j in range(len(names)):
            if i != j and row[i, j] >= 10:
                ax.text(j, i, f"{row[i, j]:.0f}", ha="center", va="center",
                        fontsize=4.5, color=C_RED, weight="bold")
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(length=0)
axa.set_ylabel("true")
save(fig, "fig_confusion")
plt.close(fig)

rowa = cm_a / cm_a.sum(1, keepdims=True) * 100
print("off-diagonal confusions >= 10% (our test set)")
for i in range(20):
    for j in range(20):
        if i != j and rowa[i, j] >= 10:
            print(f"  {CLASSES[i]:>18s} -> {CLASSES[j]:<18s} {rowa[i, j]:5.1f}%  ({cm_a[i, j]} imgs)")

## LaTeX table rows

Peak learning rates come from the training scripts: the LR in a checkpoint filename is
the warm-up initial value, not the peak.

In [ ]:
PEAK_LR = {  # (peak lr, weight decay) from the training scripts
    "standard_finetune":     (1e-4, "1e-4"),
    "conservative_finetune": (1e-5, "5e-4"),
    "aggressive_finetune":   (5e-4, "1e-4"),
    "linear_probe":          (1e-3, "1e-4"),
    "partial_freeze":        (1e-4, "1e-4"),
    "custom_wd":             (1e-4, "5e-4/0"),
    "long_warmup":           (1e-4, "1e-4"),
    "paper_rep":             (1e-4, "1e-4"),
    "bottom_freeze":         (1e-4, "1e-4"),
    "layer_decay":           (1e-4, "1e-4"),
    "bitfit":                (1e-3, "0"),
}
STRATEGY_LABEL = {
    "standard_finetune":     ("Full fine-tune (standard)", "all"),
    "conservative_finetune": ("Full fine-tune (conservative)", "all"),
    "aggressive_finetune":   ("Full fine-tune (aggressive)", "all"),
    "long_warmup":           ("Full fine-tune, 20-epoch warm-up", "all"),
    "paper_rep":             ("Paper replication (StepLR, LS 0.1)", "all"),
    "custom_wd":             ("No decay on norms/biases/GRN", "all"),
    "layer_decay":           ("Layer-wise LR decay (0.1--1.0)", "all"),
    "partial_freeze":        ("Partial freeze (stage 4 + head)", "~40\\%"),
    "bottom_freeze":         ("Bottom freeze (stages 1--2 frozen)", "~75\\%"),
    "linear_probe":          ("Linear probe (head only)", "<0.1\\%"),
    "bitfit":                ("BitFit + norms (biases, norms, head)", "0.3--0.9\\%"),
}
ORDER = ["standard_finetune", "conservative_finetune", "aggressive_finetune", "long_warmup",
         "paper_rep", "custom_wd", "layer_decay", "partial_freeze", "bottom_freeze",
         "linear_probe", "bitfit"]


def strategy_of(fname):
    stem = Path(fname).stem
    for key in sorted(PEAK_LR, key=len, reverse=True):
        if f"_{key}_pre" in stem:
            return key
    return None


init = both[both.Folder_ours == "Initial_Experiments"].copy()
init["strategy"] = init.File.map(strategy_of)
assert init.strategy.notna().all() and len(init) == 22, init[init.strategy.isna()].File.tolist()

print("% fine-tuning strategies, tiny scale")
for key in ORDER:
    label, trainable = STRATEGY_LABEL[key]
    lr, wd = PEAK_LR[key]
    cells = []
    for arch in ("GCViT", "ConvNeXtV2"):
        r = init[(init.strategy == key) & (init.arch_ours == arch)]
        cells += ([f"{r.F1_ours.iloc[0]:.2f}", f"{r.F1_prov.iloc[0]:.2f}"]
                  if len(r) else ["--", "--"])
    print(f"{label} & {trainable} & {lr:.0e} & {wd} & " + " & ".join(cells) + r" \\")

print("\n% scaling the five best recipes to base")
top5 = both[both.Folder_ours.str.startswith("Initial_Experiments_Top_5")].copy()
top5["strategy"] = top5.File.map(strategy_of)
BASE_LR = {  # (attempt 1: muP 1/width, bs 16), (attempt 2: 1/sqrt(N), bs 32)
    ("GCViT", "bitfit"):                ("1e-3",   "5.55e-4"),
    ("GCViT", "conservative_finetune"): ("5e-6",   "2.77e-6"),
    ("GCViT", "paper_rep"):             ("5e-5",   "5.55e-5"),
    ("GCViT", "aggressive_finetune"):   ("2.5e-4", "2.77e-4"),
    ("ConvNeXtV2", "conservative_finetune"): ("7.5e-6", "5.55e-6"),
}
for (arch, key), (lr_a, lr_b) in BASE_LR.items():
    naive = top5[(top5.Folder_ours.str.endswith("Failed")) & (top5.strategy == key) & (top5.arch_ours == arch)]
    mup = top5[(~top5.Folder_ours.str.endswith("Failed")) & (top5.strategy == key) & (top5.arch_ours == arch)]
    tiny = init[(init.strategy == key) & (init.arch_ours == arch)]
    print(f"{arch}-base, {STRATEGY_LABEL[key][0]} & "
          f"{tiny.F1_ours.iloc[0]:.2f} & "
          f"{lr_a} & {naive.F1_ours.iloc[0]:.2f} & {naive.Accuracy_prov.iloc[0]:.2f} & "
          f"{lr_b} & {mup.F1_ours.iloc[0]:.2f} & {mup.Accuracy_prov.iloc[0]:.2f}" + r" \\")

print("\n% trained from random initialisation")
sc = both[~both.pretrained_ours].copy()
sc["note"] = [Path(f).stem.split("_preFalse")[0].split("_", 2)[2] for f in sc.File]
sc["grn"] = np.where(sc.note.str.contains("optimized_weight_decay"), "yes", "no")
for _, r in sc.sort_values(["size_ours", "grn", "augmentation_ours"]).iterrows():
    print(f"ConvNeXtV2-{r.size_ours} & {r.batch_size_ours} & {r.grn} & {r.augmentation_ours} & "
          f"{r.Accuracy_ours:.2f} & {r.F1_ours:.2f} & {r.Accuracy_prov:.2f} & {r.F1_prov:.2f}" + r" \\")

print("\n% detector scales at the deployed operating point")
op = det[(det.confidence_threshold == 0.05) & (det.n_prompts == 14)]
for _, r in op.iterrows():
    print(f"{r.model_name.replace('.pt','').replace('-worldv2','')} & "
          f"{r['true positive']} & {r['false negative']} & {r['true negative']} & {r['false postivie']} & "
          f"{r.recall*100:.1f} & {r.precision*100:.1f} & {r.accuracy*100:.1f}" + r" \\")

print("\n% leaderboard, top 10 by F1 on our test set")
lb = both.nlargest(10, "F1_ours")
for i, (_, r) in enumerate(lb.iterrows(), 1):
    print(f"{i} & {r.arch_ours}-{r.size_ours} & {r.family.replace('&','\\&')} & "
          f"{r.Accuracy_ours:.2f} & {r.F1_ours:.2f} & {r.Accuracy_prov:.2f} & {r.F1_prov:.2f}" + r" \\")